# 06 可视化模块 (core.viz) 完整示例

覆盖 bin_plot / corr_plot / ks_plot / hist_plot / psi_plot / distribution_plot /
roc_plot / lift_plot / score_plots / strategy_plots / variable_plots 等所有图表。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import germancredit, init_setting, OptimalBinning, WOEEncoder, LogisticRegression
from hscredit.core.metrics import ks, auc
init_setting()
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
# 训练模型获取预测概率
woe = WOEEncoder(max_n_bins=5); X_woe_tr = woe.fit_transform(X_tr, y_tr); X_woe_te = woe.transform(X_te)
lr = LogisticRegression(max_iter=1000); lr.fit(X_woe_tr, y_tr)
y_prob = lr.predict_proba(X_woe_te)[:,1]
y_prob_tr = lr.predict_proba(X_woe_tr)[:,1]
print('AUC:', round(auc(y_te, y_prob),4))

## 1. bin_plot — 特征分箱图

In [ ]:
from hscredit import bin_plot
fig = bin_plot(df, feature='duration.in.month', target=target, n_bins=8, desc='贷款期限')
plt.show()

In [ ]:
# 传入分箱统计表
binner = OptimalBinning(method='mdlp', max_n_bins=5)
binner.fit(df[['credit.amount']], y)
bt = binner.bin_tables_['credit.amount']
fig2 = bin_plot(bt, desc='贷款金额')
plt.show()

In [ ]:
# 横向模式
fig3 = bin_plot(df, feature='age.in.years', target=target, orientation='horizontal')
plt.show()

## 2. corr_plot — 相关性热力图

In [ ]:
from hscredit import corr_plot
fig = corr_plot(df[num_feats[:8]], figure_size=(12,8), mask=True)
plt.show()

## 3. ks_plot — KS & ROC 曲线

In [ ]:
from hscredit import ks_plot
fig = ks_plot(y_prob, y_te, title='测试集', figsize=(14,6))
plt.show()

## 4. hist_plot — 分布直方图

In [ ]:
from hscredit import hist_plot
fig = hist_plot(df['credit.amount'], y_true=y, desc='贷款金额')
plt.show()

## 5. psi_plot — PSI稳定性图

In [ ]:
from hscredit import psi_plot
binner2 = OptimalBinning(method='quantile', max_n_bins=8)
binner2.fit(df[['credit.amount']], y)
bt_train = binner2.bin_tables_['credit.amount']
bt_test = binner2.bin_tables_['credit.amount'].copy()
bt_test['样本占比'] = bt_test['样本占比'] * np.random.uniform(0.8, 1.2, len(bt_test))
bt_test['样本占比'] /= bt_test['样本占比'].sum()
fig = psi_plot(bt_train, bt_test, desc='贷款金额', labels=['训练集','测试集'])
plt.show()

## 6. distribution_plot — 时间分布图

In [ ]:
from hscredit import distribution_plot
import datetime
np.random.seed(42)
df_ts = df.copy()
start = pd.Timestamp('2023-01-01')
df_ts['apply_date'] = [start + pd.Timedelta(days=int(d)) for d in np.random.randint(0,365,len(df_ts))]
fig = distribution_plot(df_ts, date='apply_date', target=target, freq='M')
plt.show()

## 7. roc_plot / pr_plot / lift_plot / gain_plot

In [ ]:
from hscredit import roc_plot, pr_plot, lift_plot, gain_plot
fig = roc_plot(y_te.values, y_prob); plt.show()
fig = pr_plot(y_te.values, y_prob); plt.show()
fig = lift_plot(y_te.values, y_prob); plt.show()
fig = gain_plot(y_te.values, y_prob); plt.show()

## 8. confusion_matrix_plot / calibration_plot

In [ ]:
from hscredit import confusion_matrix_plot, calibration_plot
y_pred = (y_prob > 0.3).astype(int)
fig = confusion_matrix_plot(y_te.values, y_pred); plt.show()
fig = calibration_plot(y_te.values, y_prob); plt.show()

## 9. score_dist_plot / score_bin_plot

In [ ]:
from hscredit import score_dist_plot, score_bin_plot
scores = (y_prob * 300 + 400).astype(int)
fig = score_dist_plot(scores, y_te.values); plt.show()
fig = score_bin_plot(scores, y_te.values, n_bins=10); plt.show()

## 10. 策略分析图表 (strategy_plots)

In [ ]:
from hscredit import feature_trend_by_time, feature_drift_comparison, feature_cross_heatmap
fig = feature_trend_by_time(df_ts, 'credit.amount', 'apply_date', target=target, stat='mean')
plt.show()
fig = feature_drift_comparison(X_tr, X_te, num_feats[:8]); plt.show()
fig = feature_cross_heatmap(df, 'duration.in.month', 'credit.amount', target, stat='badrate'); plt.show()

## 11. variable_plots

In [ ]:
from hscredit import variable_iv_plot, variable_missing_badrate_plot
fig = variable_iv_plot(df, num_feats, target, top_n=10); plt.show()
fig = variable_missing_badrate_plot(df, num_feats, target); plt.show()

## 12. score_plots (KS / LIFT 曲线)

In [ ]:
from hscredit import score_ks_plot, score_lift_plot, score_approval_badrate_curve
fig = score_ks_plot(datasets={'训练集':(y_tr.values,y_prob_tr),'测试集':(y_te.values,y_prob)}); plt.show()
fig = score_lift_plot(y_te.values, y_prob); plt.show()
fig = score_approval_badrate_curve(y_te.values, y_prob); plt.show()

## 13. plot_weights — 逻辑回归系数图

In [ ]:
from hscredit import plot_weights
lr2 = LogisticRegression(calculate_stats=True, max_iter=1000)
lr2.fit(X_woe_tr, y_tr)
fig = plot_weights(lr2); plt.show()